# 02 - Multi-Agent System

Multi-agent system with specialist agents, orchestrator, communication, and voting.


In [ ]:
import numpy as np
import re
import math
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import random
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')


## 1. Base Agent Class


In [ ]:
class BaseAgent:
    def __init__(self, name, role):
        self.name = name
        self.role = role
        self.memory = []
        self.message_queue = []
    def receive_message(self, sender, message):
        self.message_queue.append({'from': sender, 'content': message})
    def send_message(self, recipient, message, all_agents):
        for agent in all_agents:
            if agent.name == recipient:
                agent.receive_message(self.name, message)
                return True
        return False
    def broadcast(self, message, all_agents):
        for agent in all_agents:
            if agent.name != self.name:
                agent.receive_message(self.name, message)
    def process(self, task):
        raise NotImplementedError
    def get_status(self):
        return {'name': self.name, 'role': self.role, 'memory_size': len(self.memory), 'pending_messages': len(self.message_queue)}

print('BaseAgent defined!')


## 2. Specialist Agents


In [ ]:
class MathAgent(BaseAgent):
    def __init__(self, name):
        super().__init__(name, 'Mathematics')
    def process(self, task):
        numbers = re.findall(r'\d+', task)
        ops = re.findall(r'[+\-*/]', task)
        if numbers and ops:
            try:
                result = eval(''.join([n for pair in zip(numbers, ops + ['']) for n in pair]))
                return f'{self.name}: Result = {result}'
            except:
                return f'{self.name}: Could not calculate'
        return f'{self.name}: No math detected'

class SearchAgent(BaseAgent):
    def __init__(self, name, knowledge_base):
        super().__init__(name, 'Search')
        self.kb = knowledge_base
    def process(self, task):
        task_lower = task.lower()
        for key, value in self.kb.items():
            if key in task_lower or all(w in task_lower for w in key.split()):
                return f'{self.name}: Found - {value}'
        return f'{self.name}: No results found'

class LogicAgent(BaseAgent):
    def __init__(self, name):
        super().__init__(name, 'Logic')
    def process(self, task):
        if 'if' in task.lower() and 'then' in task.lower():
            return f'{self.name}: This is a conditional statement. Evaluating... Valid logic structure.'
        if 'all' in task.lower() or 'some' in task.lower():
            return f'{self.name}: This involves quantifiers. Analyzing scope...'
        return f'{self.name}: Basic logic analysis complete'

class CreativeAgent(BaseAgent):
    def __init__(self, name):
        super().__init__(name, 'Creative')
    def process(self, task):
        ideas = ['Innovative approach', 'Novel solution', 'Creative insight', 'Fresh perspective']
        return f'{self.name}: {random.choice(ideas)} for "{task[:30]}..."'

print('Specialist agents defined!')


## 3. Orchestrator


In [ ]:
class Orchestrator:
    def __init__(self):
        self.agents = []
        self.task_history = []
    def register_agent(self, agent):
        self.agents.append(agent)
    def route_task(self, task):
        task_lower = task.lower()
        if any(w in task_lower for w in ['calculate', 'math', 'sum', 'multiply', 'divide']):
            return [a for a in self.agents if a.role == 'Mathematics']
        elif any(w in task_lower for w in ['find', 'search', 'what is', 'who is', 'where']):
            return [a for a in self.agents if a.role == 'Search']
        elif any(w in task_lower for w in ['if', 'then', 'logic', 'prove', 'all', 'some']):
            return [a for a in self.agents if a.role == 'Logic']
        else:
            return [a for a in self.agents if a.role == 'Creative']
    def execute(self, task):
        print(f'\nOrchestrator received: {task}')
        selected = self.route_task(task)
        print(f'Selected agents: {[a.name for a in selected]}')
        results = []
        for agent in selected:
            result = agent.process(task)
            results.append((agent.name, result))
            agent.memory.append((task, result))
            print(f'  {result}')
        self.task_history.append({'task': task, 'agents': [a.name for a in selected], 'results': results})
        return results

kb = {'paris': 'Capital of France', 'berlin': 'Capital of Germany', 'tokyo': 'Capital of Japan'}
orch = Orchestrator()
orch.register_agent(MathAgent('MathBot'))
orch.register_agent(SearchAgent('SearchBot', kb))
orch.register_agent(LogicAgent('LogicBot'))
orch.register_agent(CreativeAgent('CreativeBot'))
print('Orchestrator ready!')


In [ ]:
tasks = [
    'Calculate 15 * 23 + 7',
    'What is the capital of France?',
    'If all birds can fly and penguins are birds, then penguins can fly',
    'Generate ideas for a new app'
]
for task in tasks:
    orch.execute(task)
    print()


## 4. Communication Protocol


In [ ]:
# Demonstrate agent communication
math_bot = orch.agents[0]
search_bot = orch.agents[1]

math_bot.send_message('SearchBot', 'Please find the value of pi', orch.agents)
print(f'Messages to SearchBot: {search_bot.message_queue}')

# Process messages
for msg in search_bot.message_queue:
    print(f'{msg["from"]} -> SearchBot: {msg["content"]}')

search_bot.broadcast('I found pi = 3.14159', orch.agents)
for agent in orch.agents:
    if agent.name != 'SearchBot':
        print(f'{agent.name} received: {agent.message_queue[-1] if agent.message_queue else None}')


## 5. Voting Mechanism


In [ ]:
class VotingSystem:
    def __init__(self, agents):
        self.agents = agents
    def vote(self, task, options):
        votes = defaultdict(int)
        for agent in self.agents:
            # Simple voting: random choice weighted by role relevance
            if agent.role == 'Mathematics' and any(op in task for op in ['+', '-', '*', '/']):
                choice = options[0]  # Math agent prefers first option
            elif agent.role == 'Logic' and 'if' in task.lower():
                choice = options[-1]  # Logic agent prefers last option
            else:
                choice = random.choice(options)
            votes[choice] += 1
            print(f'  {agent.name} ({agent.role}) voted: {choice}')
        winner = max(votes, key=votes.get)
        return winner, votes

vs = VotingSystem(orch.agents)
options = ['Option A: Quick fix', 'Option B: Thorough analysis', 'Option C: Creative workaround']
winner, vote_counts = vs.vote('How to solve this bug?', options)
print(f'\nWinner: {winner}')
print(f'Vote counts: {dict(vote_counts)}')


## 6. System Architecture Visualization


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Orchestrator
ax.add_patch(plt.Circle((5, 8), 0.8, color='red', alpha=0.7))
ax.text(5, 8, 'Orchestrator', ha='center', va='center', fontweight='bold', color='white')

# Agents
agent_positions = [(2, 5), (5, 5), (8, 5), (5, 2)]
agent_names = ['MathBot', 'SearchBot', 'LogicBot', 'CreativeBot']
colors = ['blue', 'green', 'purple', 'orange']
for pos, name, color in zip(agent_positions, agent_names, colors):
    ax.add_patch(plt.Circle(pos, 0.7, color=color, alpha=0.7))
    ax.text(pos[0], pos[1], name, ha='center', va='center', fontweight='bold', color='white', fontsize=9)
    ax.plot([5, pos[0]], [8, pos[1]], 'k--', alpha=0.3)

# Communication lines
for i, pos1 in enumerate(agent_positions):
    for pos2 in agent_positions[i+1:]:
        ax.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], 'gray', alpha=0.2, linewidth=1)

ax.set_title('Multi-Agent System Architecture', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Exercises


In [ ]:
# Exercise 1: Add conflict resolution
# Exercise 2: Implement bidding system
# Exercise 3: Add reputation tracking
# Exercise 4: Build hierarchical agents
print('Exercises listed!')
